# SCRIPTOR — Prototipo Funcional TRL5
**Universidad Nacional Abierta y a Distancia – UNAD**  
Escuela de Ciencias Básicas, Tecnología e Ingeniería – ECBTI  
Curso: Proyecto de Grado | Código: 202016907 | Grupo 2  

**Autores:** Raquel Sofía Naranjo Gaviria · Héctor Hernán Ramírez Campo · José Alejandro Contreras Mantilla  

---
### ¿Qué hace este notebook?
SCRIPTOR convierte documentos PDF e imágenes (JPG/PNG) en audio y texto accesible para personas con discapacidad visual.

**Pipeline de 4 módulos:**
1. Módulo 1 — Ingesta y Preprocesamiento
2. Módulo 2 — Motor OCR (Tesseract)
3. Módulo 3 — Procesamiento NLP
4. Módulo 4 — Salida Accesible (MP3 + TXT)

> **Instrucciones:** Ejecuta las celdas en orden de arriba a abajo.

## CELDA 1 — Instalación de dependencias
Instala Tesseract OCR, poppler y todas las bibliotecas de Python necesarias. Ejecutar solo una vez por sesión.

In [1]:
# Instala Tesseract OCR y el paquete de idioma español
!apt-get install -y tesseract-ocr tesseract-ocr-spa

# Instala poppler-utils para convertir páginas PDF a imágenes
!apt-get install -y poppler-utils

# Instala ffmpeg para manejo de audio
!apt-get install -y ffmpeg

# Instala las bibliotecas Python:
# pytesseract  → conecta Python con Tesseract OCR
# pdf2image    → convierte PDFs a imágenes PIL
# Pillow       → manipulación de imágenes
# opencv-python-headless → preprocesamiento avanzado de imágenes
# nltk         → procesamiento de lenguaje natural
# gTTS         → convierte texto a audio MP3 (Google TTS)
# pydub        → manipulación de archivos de audio
!pip install pytesseract pdf2image Pillow opencv-python-headless nltk gTTS pydub --quiet

print('Todas las dependencias instaladas correctamente.')

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
The following NEW packages will be installed:
  tesseract-ocr-spa
0 upgraded, 1 newly installed, 0 to remove and 3 not upgraded.
Need to get 951 kB of archives.
After this operation, 2,309 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-spa all 1:4.00~git30-7274cfa-1.1 [951 kB]
Fetched 951 kB in 0s (10.1 MB/s)
Selecting previously unselected package tesseract-ocr-spa.
(Reading database ... 118251 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-spa_1%3a4.00~git30-7274cfa-1.1_all.deb ...
Unpacking tesseract-ocr-spa (1:4.00~git30-7274cfa-1.1) ...
Setting up tesseract-ocr-spa (1:4.00~git30-7274cfa-1.1) ...
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will

## CELDA 2 — Importación y configuración
Carga las bibliotecas y configura Tesseract en español.

In [2]:
# Bibliotecas del sistema
import os           # Manejo de rutas y carpetas
import re           # Expresiones regulares para limpiar texto
import nltk
nltk.download('punkt_tab')
# Procesamiento de imágenes
from PIL import Image                        # Manipulación de imágenes
import cv2                                   # OpenCV: binarización y corrección
import numpy as np                           # Arrays de píxeles

# OCR
import pytesseract                           # Interfaz Python para Tesseract
from pdf2image import convert_from_path      # Convierte PDF a imágenes PIL

# Procesamiento de lenguaje natural
import nltk
nltk.download('punkt', quiet=True)           # Tokenizador de oraciones
nltk.download('stopwords', quiet=True)       # Palabras comunes en español
from nltk.tokenize import sent_tokenize      # Divide texto en oraciones

# Audio
from gtts import gTTS                        # Google Text-to-Speech
from IPython.display import Audio, display   # Reproductor de audio en Colab

# Subida de archivos en Colab
from google.colab import files

# Ruta de Tesseract en el servidor de Colab
pytesseract.pytesseract.tesseract_cmd = '/usr/bin/tesseract'

# Configuración de Tesseract:
# --oem 3  → mejor motor disponible (LSTM + legacy)
# --psm 3  → segmentación automática de página
# -l spa   → idioma español
TESSERACT_CONFIG = '--oem 3 --psm 3 -l spa'

# Carpeta donde se guardarán los archivos de salida
os.makedirs('/content/scriptor_output', exist_ok=True)

print('Bibliotecas importadas y configuración lista.')
print(f'Tesseract version: {pytesseract.get_tesseract_version()}')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Bibliotecas importadas y configuración lista.
Tesseract version: 4.1.1


## CELDA 3 — MÓDULO 1: Ingesta y Preprocesamiento
Carga el documento y aplica binarización, reducción de ruido y corrección de imagen para mejorar el OCR.

In [3]:
def preprocesar_imagen(imagen_pil):
    """
    Recibe una imagen PIL y aplica mejoras para optimizar el OCR.
    Retorna la imagen mejorada como objeto PIL.Image.
    """
    # Convierte imagen PIL a array numpy (formato que usa OpenCV)
    img_np = np.array(imagen_pil)

    # Paso 1: Convertir a escala de grises
    # El color no aporta al OCR y aumenta el tiempo de procesamiento
    if len(img_np.shape) == 3:
        gris = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    else:
        gris = img_np

    # Paso 2: Binarización Otsu
    # Convierte la imagen a blanco y negro puro
    # Otsu calcula automáticamente el umbral óptimo de separación
    _, binaria = cv2.threshold(
        gris, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    # Paso 3: Reducción de ruido con filtro de mediana
    # Elimina píxeles aislados sin borrar los bordes del texto
    sin_ruido = cv2.medianBlur(binaria, 3)

    # Paso 4: Dilatación morfológica
    # Engrosa levemente los caracteres para que Tesseract los detecte mejor
    kernel = np.ones((2, 1), np.uint8)
    mejorada = cv2.dilate(sin_ruido, kernel, iterations=1)

    # Convierte de vuelta a PIL.Image para usar con pytesseract
    return Image.fromarray(mejorada)


def cargar_documento(ruta_archivo):
    """
    Detecta si el archivo es PDF o imagen, lo carga y preprocesa.
    Retorna una lista de imágenes PIL (una por página).
    """
    extension = os.path.splitext(ruta_archivo)[1].lower()
    paginas_raw = []

    if extension == '.pdf':
        # convert_from_path convierte cada página del PDF en imagen PIL
        # dpi=300 genera imágenes de alta resolución (mejor para OCR)
        print(f'Cargando PDF: {os.path.basename(ruta_archivo)}')
        paginas_raw = convert_from_path(ruta_archivo, dpi=300)
        print(f'  {len(paginas_raw)} pagina(s) detectada(s)')

    elif extension in ['.jpg', '.jpeg', '.png', '.bmp', '.tiff']:
        # Carga la imagen y la trata como documento de 1 sola página
        print(f'Cargando imagen: {os.path.basename(ruta_archivo)}')
        imagen = Image.open(ruta_archivo).convert('RGB')
        paginas_raw = [imagen]

    else:
        raise ValueError(f'Formato no soportado: {extension}. Use PDF, JPG o PNG.')

    # Preprocesa cada página para mejorar la calidad antes del OCR
    print('Aplicando preprocesamiento de imagen...')
    paginas_procesadas = []
    for i, pagina in enumerate(paginas_raw):
        pagina_mejorada = preprocesar_imagen(pagina)
        paginas_procesadas.append(pagina_mejorada)
        print(f'  Pagina {i+1} preprocesada')

    return paginas_procesadas


print('Modulo 1 (Ingesta y Preprocesamiento) cargado.')

Modulo 1 (Ingesta y Preprocesamiento) cargado.


## CELDA 4 — MÓDULO 2: Motor OCR
Usa Tesseract OCR v5 configurado en español para extraer el texto de cada página.

In [4]:
def extraer_texto_ocr(paginas_procesadas):
    """
    Aplica Tesseract OCR sobre cada página preprocesada.
    Retorna el texto completo y la confianza promedio del reconocimiento.
    """
    textos_por_pagina = []   # Texto extraído de cada página
    confianzas = []          # Nivel de confianza de cada página (0-100)

    print('Extrayendo texto con Tesseract OCR...')

    for i, pagina in enumerate(paginas_procesadas):

        # image_to_data retorna texto Y datos de confianza por palabra
        datos = pytesseract.image_to_data(
            pagina,
            config=TESSERACT_CONFIG,
            output_type=pytesseract.Output.DICT
        )

        # image_to_string extrae solo el texto (más directo)
        texto_pagina = pytesseract.image_to_string(
            pagina,
            config=TESSERACT_CONFIG
        )

        # Calcula confianza promedio (ignora valores -1 que son espacios)
        confianzas_validas = [c for c in datos['conf'] if c >= 0]
        confianza_pagina = (
            sum(confianzas_validas) / len(confianzas_validas)
            if confianzas_validas else 0
        )

        textos_por_pagina.append(texto_pagina)
        confianzas.append(confianza_pagina)

        palabras = len(texto_pagina.split())
        print(f'  Pagina {i+1}: {palabras} palabras | Confianza: {confianza_pagina:.1f}%')

    # Une el texto de todas las páginas con doble salto de línea entre ellas
    texto_completo = '\n\n'.join(textos_por_pagina)
    confianza_promedio = sum(confianzas) / len(confianzas) if confianzas else 0

    print(f'\nConfianza OCR promedio: {confianza_promedio:.1f}%')
    return texto_completo, confianza_promedio


print('Modulo 2 (Motor OCR) cargado.')

Modulo 2 (Motor OCR) cargado.


## CELDA 5 — MÓDULO 3: Procesamiento NLP
Limpia el texto crudo del OCR: elimina guiones, espacios dobles, caracteres extraños y líneas cortas.

In [5]:
def limpiar_texto(texto_crudo):
    """
    Limpia el texto extraído por OCR usando expresiones regulares.
    Retorna el texto corregido y listo para síntesis de voz.
    """
    texto = texto_crudo

    # Une palabras partidas con guion al final de línea: 'infor-\nmación' → 'información'
    texto = re.sub(r'(\w)-\n(\w)', r'\1\2', texto)

    # Reemplaza saltos de línea simples dentro de un párrafo por espacios
    # Conserva los dobles saltos (que separan párrafos)
    texto = re.sub(r'(?<!\n)\n(?!\n)', ' ', texto)

    # Elimina espacios y tabulaciones múltiples consecutivos
    texto = re.sub(r'[ \t]+', ' ', texto)

    # Elimina el carácter de fin de página que generan los PDFs
    texto = texto.replace('\x0c', '')

    # Compacta más de dos saltos de línea seguidos a máximo dos
    texto = re.sub(r'\n{3,}', '\n\n', texto)

    # Elimina espacios al inicio y final de cada línea
    lineas = [linea.strip() for linea in texto.split('\n')]
    texto = '\n'.join(lineas)

    # Elimina líneas muy cortas (menos de 3 caracteres): son artefactos OCR
    lineas = [l for l in texto.split('\n') if len(l.strip()) >= 3 or l.strip() == '']
    texto = '\n'.join(lineas)

    # Elimina espacios antes de puntuación: 'hola , mundo' → 'hola, mundo'
    texto = re.sub(r'\s+([.,;:!?])', r'\1', texto)

    return texto.strip()


def estructurar_texto(texto_limpio):
    """
    Calcula estadísticas básicas del texto procesado.
    Retorna un diccionario con palabras, párrafos, oraciones y duración estimada.
    """
    parrafos = [p.strip() for p in texto_limpio.split('\n\n') if p.strip()]
    total_palabras = len(texto_limpio.split())
    # sent_tokenize divide el texto en oraciones usando el idioma español
    oraciones = sent_tokenize(texto_limpio, language='spanish')

    return {
        'parrafos': len(parrafos),
        'oraciones': len(oraciones),
        'palabras': total_palabras,
        'tiempo_audio_min': round(total_palabras / 200, 1)  # 200 palabras/min promedio
    }


print('Modulo 3 (Procesamiento NLP) cargado.')

Modulo 3 (Procesamiento NLP) cargado.


## CELDA 6 — MÓDULO 4: Salida Accesible (TTS + TXT)
Genera el archivo MP3 con síntesis de voz y el archivo TXT con codificación UTF-8 compatible con lectores de pantalla.

In [6]:
def scriptor_pipeline(ruta_archivo):
    """
    Pipeline completo de SCRIPTOR.
    Procesa un documento y genera las salidas accesibles (MP3 + TXT).
    Retorna un diccionario con texto, rutas de archivos y estadísticas.
    """
    nombre_base = os.path.splitext(os.path.basename(ruta_archivo))[0]

    print('=' * 55)
    print('       SCRIPTOR — INICIO DE PROCESAMIENTO')
    print('=' * 55)
    print(f'Documento: {os.path.basename(ruta_archivo)}\n')

    # MÓDULO 1: Ingesta y Preprocesamiento
    print('[MÓDULO 1] Ingesta y Preprocesamiento...')
    paginas = cargar_documento(ruta_archivo)
    print(f'           {len(paginas)} pagina(s) lista(s)\n')

    # MÓDULO 2: Motor OCR
    print('[MÓDULO 2] Extraccion de texto con OCR...')
    texto_crudo, confianza = extraer_texto_ocr(paginas)
    print(f'           Caracteres extraidos: {len(texto_crudo)}\n')

    # Verifica que se haya extraído contenido
    if len(texto_crudo.strip()) < 10:
        print('ADVERTENCIA: Se extrajo muy poco texto. El documento puede estar vacío.')
        return None

    # MÓDULO 3: Procesamiento NLP
    print('[MÓDULO 3] Limpieza y estructuracion del texto...')
    texto_limpio = limpiar_texto(texto_crudo)
    stats = estructurar_texto(texto_limpio)
    print(f"           {stats['palabras']} palabras | {stats['parrafos']} parrafos | {stats['oraciones']} oraciones")
    print(f"           Duracion estimada del audio: ~{stats['tiempo_audio_min']} minutos\n")

    # MÓDULO 4: Salida Accesible
    print('[MÓDULO 4] Generando salidas accesibles...')
    ruta_audio = generar_audio(texto_limpio, nombre_salida=f'{nombre_base}_audio')
    ruta_txt   = guardar_texto(texto_limpio, nombre_salida=f'{nombre_base}_texto')

    print('\n' + '=' * 55)
    print('       SCRIPTOR — PROCESAMIENTO COMPLETADO')
    print('=' * 55)
    print(f"  Confianza OCR : {confianza:.1f}%")
    print(f"  Palabras      : {stats['palabras']}")
    print(f"  Audio         : {ruta_audio}")
    print(f"  Texto         : {ruta_txt}")
    print('=' * 55)

    return {
        'texto': texto_limpio,
        'confianza_ocr': confianza,
        'ruta_audio': ruta_audio,
        'ruta_txt': ruta_txt,
        'estadisticas': stats
    }


print('Pipeline principal cargado. SCRIPTOR esta listo para usar.')

Pipeline principal cargado. SCRIPTOR esta listo para usar.


## CELDA 8 — Subir tu documento y ejecutar SCRIPTOR
Sube un archivo PDF o imagen desde tu computador y procésalo con el pipeline completo.

In [ ]:
print('Selecciona el archivo PDF o imagen que deseas procesar...')

# files.upload() abre el explorador de archivos de tu computador
# Retorna un diccionario: {nombre_archivo: contenido_en_bytes}
archivos_subidos = files.upload()

resultados = {}  # Guarda el resultado de cada archivo procesado

for nombre_archivo, contenido in archivos_subidos.items():

    # Guarda el archivo en el servidor de Colab
    ruta_guardada = f'/content/{nombre_archivo}'
    with open(ruta_guardada, 'wb') as f:  # 'wb' = escritura en modo binario
        f.write(contenido)

    print(f'Archivo recibido: {nombre_archivo} ({len(contenido)/1024:.1f} KB)')

    # Ejecuta el pipeline completo de SCRIPTOR
    resultado = scriptor_pipeline(ruta_guardada)
    if resultado:
        resultados[nombre_archivo] = resultado

print(f'\nProcesamiento completado para {len(resultados)} archivo(s).')

Selecciona el archivo PDF o imagen que deseas procesar...


## CELDA 9 — Reproducir el audio generado
Reproduce el archivo MP3 directamente dentro del notebook.

In [ ]:
# Reproduce el audio del primer archivo procesado
if resultados:
    primer = list(resultados.values())[0]
    print(f"Reproduciendo: {primer['ruta_audio']}")
    # autoplay=False: el usuario debe darle play manualmente
    display(Audio(primer['ruta_audio'], autoplay=False))
else:
    print('No hay audio disponible. Ejecuta primero la Celda 8.')

## CELDA 10 — Ver el texto extraído
Muestra el texto limpio en pantalla.

In [ ]:
if resultados:
    primer = list(resultados.values())[0]
    texto  = primer['texto']
    stats  = primer['estadisticas']

    print('=' * 55)
    print('         TEXTO EXTRAÍDO POR SCRIPTOR')
    print('=' * 55)
    print(f"Palabras: {stats['palabras']} | Parrafos: {stats['parrafos']} | Oraciones: {stats['oraciones']}")
    print('=' * 55)
    print()

    PREVIEW = 3000  # Muestra los primeros 3000 caracteres
    if len(texto) <= PREVIEW:
        print(texto)
    else:
        print(texto[:PREVIEW])
        print(f'\n... [{len(texto) - PREVIEW} caracteres mas] ...')
        print(f"Texto completo en: {primer['ruta_txt']}")
else:
    print('No hay texto disponible. Ejecuta primero la Celda 8.')

## CELDA 11 — Descargar los archivos generados
Descarga el MP3 y el TXT a tu computador.

In [ ]:
if resultados:
    primer = list(resultados.values())[0]
    print('Descargando archivos generados por SCRIPTOR...')
    # files.download() abre el dialogo de descarga del navegador
    print('Descargando archivo de audio (MP3)...')
    files.download(primer['ruta_audio'])
    print('Descargando archivo de texto (TXT)...')
    files.download(primer['ruta_txt'])
    print('Descarga completada.')
else:
    print('No hay archivos. Ejecuta primero la Celda 8.')

## CELDA 12 — Prueba rápida sin subir archivo
Prueba SCRIPTOR con un texto de ejemplo para verificar que todo funciona correctamente.

In [ ]:
# Texto de ejemplo: simula un documento gubernamental colombiano
texto_ejemplo = (
    'MINISTERIO DE TECNOLOGIAS DE LA INFORMACION Y LAS COMUNICACIONES\n'
    'Republica de Colombia\n\n'
    'RESOLUCION NUMERO 2021-001\n\n'
    'Por la cual se establecen lineamientos de accesibilidad digital '
    'para entidades del Estado colombiano.\n\n'
    'El Ministro de Tecnologias de la Informacion y las Comunicaciones, '
    'en ejercicio de sus facultades legales, y en especial las conferidas '
    'por la Ley 1712 de 2014 y el Decreto 1078 de 2015,\n\n'
    'RESUELVE:\n\n'
    'Articulo 1. Objeto. La presente resolucion tiene por objeto establecer '
    'los lineamientos tecnicos que deben seguir todas las entidades publicas '
    'colombianas para garantizar la accesibilidad digital de sus plataformas.\n\n'
    'Articulo 2. Ambito de aplicacion. Las disposiciones de la presente '
    'resolucion aplican a todas las entidades de la Rama Ejecutiva del '
    'poder publico del orden nacional.'
)

print('Ejecutando prueba rapida con texto de ejemplo...')
print(f'Palabras en el texto de prueba: {len(texto_ejemplo.split())}\n')

# Limpia y estructura el texto de ejemplo con el Módulo 3
texto_limpio_ej = limpiar_texto(texto_ejemplo)
stats_ej = estructurar_texto(texto_limpio_ej)
print(f"Texto limpio: {stats_ej['palabras']} palabras | {stats_ej['parrafos']} parrafos\n")

# Genera el audio con el Módulo 4
ruta_audio_ej = generar_audio(texto_limpio_ej, nombre_salida='ejemplo_audio')
ruta_txt_ej   = guardar_texto(texto_limpio_ej, nombre_salida='ejemplo_texto')

print('\nReproduciendo audio de ejemplo:')
# Reproduce el audio generado directamente en el notebook
display(Audio(ruta_audio_ej, autoplay=False))

print('\nPrueba completada. SCRIPTOR funciona correctamente.')

---
## Resumen del Prototipo SCRIPTOR

| Módulo | Tecnología | Función |
|--------|-----------|--------|
| 1 – Ingesta | pdf2image, Pillow, OpenCV | Carga PDF/imagen y mejora calidad |
| 2 – OCR | Tesseract v5, pytesseract | Extrae texto en español |
| 3 – NLP | NLTK, expresiones regulares | Limpia y estructura el texto |
| 4 – Salida | gTTS, UTF-8 | Genera MP3 y TXT accesibles |

**Nivel TRL:** TRL5 — prototipo validado en entorno relevante  
**Lenguaje:** Python 3.10 | **Entorno:** Google Colab / Ubuntu Linux  
**Licencia:** MIT